# 看懂五大模組的共同規範

這章不先寫複雜功能，而是看清楚 Agentic-SDK 如何判斷一個物件能不能被放進 workflow。五大模組共用同一個執行標準：讀取 `WorkflowState`，回傳 `ModuleOutput`。

## 在 Colab 準備環境

如果你是在 Colab 開啟，先複製專案並切換到專案資料夾；如果你已經在專案資料夾，可以直接跳過這格。

In [ ]:
!git clone https://github.com/R300-AI/Agentic-SDK.git
%cd Agentic-SDK

## 載入規範會用到的型別

這章會直接使用 SDK 公開的狀態與輸出型別，讓模組標準可以被看見，而不是藏在內建模組裡。

In [ ]:
from agentic_sdk import Workflow
from agentic_sdk.core import ContextEntry, ContextEntryType, ModuleOutput, WorkflowState

## 共同規範

五大模組都遵守同一個最小介面：

- `name`：模組在 workflow 裡的階段名稱
- `__call__(state)`：workflow 執行模組時呼叫的方法
- `WorkflowState`：目前這次執行的狀態
- `ModuleOutput`：模組回傳給 workflow 的結果

`Workflow` 不要求模組一定要繼承 base class。只要物件符合這個形狀，就可以被放進對應欄位。

## 五大模組的責任

| 模組 | 常見 `name` | 主要責任 |
|---|---|---|
| perceive | `perceive` | 整理使用者輸入 |
| plan | `plan` | 決定下一步 |
| retrieve | `retrieve` | 取得參考內容 |
| action | `action` | 產生回答或執行動作 |
| reflect | `reflect` | 檢查結果並決定是否通過 |

這些模組的 Python 介面相同；差別在於每個階段要讀什麼、寫什麼，以及下一站要去哪裡。

## `ModuleOutput` 的三個欄位

`next_module` 決定下一站；如果是 `None`，流程結束。

`payload` 會寫入 `state.entities`，讓後面的模組可以讀到。

`context_updates` 會留下這一步的紀錄，讓結果、除錯、Reflect 或 UI 可以觀察流程。

## 用最小 action 驗證規範

下面只寫一個最小 action。它不示範複雜功能，只示範一個模組如何符合 `name`、`__call__(state)` 與 `ModuleOutput` 這三個標準。

In [ ]:
class FixedAction:
    name = 'action'

    def __call__(self, state: WorkflowState) -> ModuleOutput:
        message = f'收到問題：{state.latest_user_message()}'
        return ModuleOutput(
            next_module=None,
            payload={'latest_final_message': message},
            context_updates=[
                ContextEntry(
                    type=ContextEntryType.ACTION_RESULT,
                    content=message,
                    metadata={'source': 'fixed_action'},
                )
            ],
        )

## 建立只執行 action 的 workflow

這裡讓 workflow 從 action 開始，避免預設 retrieve 介入。這樣輸出只反映 `FixedAction` 是否符合模組規範。

In [ ]:
workflow = Workflow(
    workflow_name='模組規範示範 Agent',
    entry_module='action',
    action=FixedAction(),
)

## 執行一次

執行後先看最後回覆。因為 `FixedAction` 回傳 `next_module=None`，所以 action 跑完後流程就結束。

In [ ]:
result = workflow.run('請示範五大模組的共同規範。')
print(result.final_message)

## 檢查模組交回 workflow 的資料

`payload` 會出現在 `result.entities`。`context_updates` 會出現在 `result.entries`。這兩個位置讓後續模組、除錯工具或前端介面能用同一種方式觀察流程。

In [ ]:
print('entities:', result.entities)

for entry in result.entries:
    print(entry.type, entry.content, entry.metadata)

## 這一章的重點

五大模組共用同一個執行規範：

- `name` 定義模組階段
- `__call__(state)` 是 workflow 執行入口
- `WorkflowState` 是讀取目前狀態的地方
- `ModuleOutput.next_module` 決定下一站
- `ModuleOutput.payload` 把資料交給後續模組
- `ModuleOutput.context_updates` 留下可追蹤紀錄

先理解這個共同規範，再去寫 perceive、plan、retrieve、action 或 reflect，才不會把五大模組想成五套不同 API。